# Feature Engineering v3

## Objective

This notebook builds the third iteration of the modeling dataset by extending the v2 feature set with richer spatial, directional, and temporal features.

The goals are to:

- improve representation of player motion,
- capture directional and target-relative information,
- introduce simple frame-to-frame temporal features,
- and prepare a stronger dataset for XGBoost training.

## Motivation

Evaluation of earlier models suggested that prediction errors remain higher in complex situations and that the current feature set does not fully capture motion dynamics or temporal context.

This version aims to address those limitations through more expressive engineered features.

In [1]:
import sys 
from pathlib import Path 

import pandas as pd
import numpy as np 

sys.path.append(str(Path("..").resolve()))

from src.config import PROCESSED_DIR
pd.set_option("display.max_columns", None)

## Load v2 Dataset

We begin by loading the v2 processed dataset so that new feature engineering can be layered on top of the existing baseline and spatial feature set.

In [2]:
input_path= PROCESSED_DIR/"baseline_train_v2.parquet"
df=pd.read_parquet(input_path)

print("Loaded v2 dataset: ",df.shape)
display(df.head())

Loaded v2 dataset:  (152305, 32)


,game_id,play_id,nfl_id,player_name,frame_id,x,y,s,a,dir,o,absolute_yardline_number,player_weight,player_height_inches,player_age,is_moving_right,player_position,player_side,player_role,ball_land_x,ball_land_y,num_frames_output,dist_to_ball,speed_x,speed_y,acc_vector,x_centered,y_centered,momentum,speed_times_dir,x_times_speed,speed_bin_code
0,2023090700,101,46137,Justin Reid,1,51.32,20.69,0.31,0.49,79.43,267.68,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21,24.078863,0.056865,0.304740,0.49,-8.68,-5.96,63.24,24.6233,15.9092,0
1,2023090700,101,46137,Justin Reid,2,51.35,20.66,0.36,0.74,118.07,268.66,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21,24.037938,-0.169398,0.317654,0.74,-8.65,-5.99,73.44,42.5052,18.4860,0
2,2023090700,101,46137,Justin Reid,3,51.39,20.63,0.44,0.76,130.89,269.78,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21,23.992069,-0.288028,0.332626,0.76,-8.61,-6.02,89.76,57.5916,22.6116,0
3,2023090700,101,46137,Justin Reid,4,51.43,20.61,0.48,0.62,134.50,269.78,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21,23.954911,-0.336436,0.342360,0.62,-8.57,-6.04,97.92,64.5600,24.6864,0
4,2023090700,101,46137,Justin Reid,5,51.48,20.58,0.54,0.44,129.79,269.06,42,204,73,26,1,SS,Defense,Defensive Coverage,63.259998,-0.22,21,23.904149,-0.345587,0.414933,0.44,-8.52,-6.07,110.16,70.0866,27.7992,0


In [3]:
print("Current columns: ")
print(df.columns.tolist())

Current columns: 
['game_id', 'play_id', 'nfl_id', 'player_name', 'frame_id', 'x', 'y', 's', 'a', 'dir', 'o', 'absolute_yardline_number', 'player_weight', 'player_height_inches', 'player_age', 'is_moving_right', 'player_position', 'player_side', 'player_role', 'ball_land_x', 'ball_land_y', 'num_frames_output', 'dist_to_ball', 'speed_x', 'speed_y', 'acc_vector', 'x_centered', 'y_centered', 'momentum', 'speed_times_dir', 'x_times_speed', 'speed_bin_code']


## Add Target-Relative Spatial Features

We create features that describe the players's position relative to the ball landing location.

These features help the model understand:

    -how far the player is from the target.
    -whether the player is ahead or behind the target.
    -whether the player is laterally aligned with the target.

In [4]:
df["dx_to_ball"]=df["ball_land_x"]-df["x"]
df["dy_to_ball"]=df["ball_land_y"]-df["y"]

df["dist_to_ball"]= np.sqrt(
    df["dx_to_ball"]**2 + df["dy_to_ball"]**2 
)
display(df[["x","y","ball_land_x","ball_land_y","dx_to_ball","dy_to_ball", "dist_to_ball"]].head())

,x,y,ball_land_x,ball_land_y,dx_to_ball,dy_to_ball,dist_to_ball
0,51.32,20.69,63.259998,-0.22,11.939998,-20.91,24.078863
1,51.35,20.66,63.259998,-0.22,11.909998,-20.88,24.037938
2,51.39,20.63,63.259998,-0.22,11.869998,-20.85,23.992069
3,51.43,20.61,63.259998,-0.22,11.829998,-20.83,23.954911
4,51.48,20.58,63.259998,-0.22,11.779998,-20.80,23.904149


## Add Directional Movement Features

We decompose player speed into horizontal and vertical components using the movement direction. This helps the model represent not only how fast the player is moving, but also where that movement is directed.

In [5]:
dir_rad=np.deg2rad(df["dir"])

df["speed_x"]=df["s"]*np.cos(dir_rad)
df["speed_y"]=df["s"]*np.sin(dir_rad)

display(df[["s","dir","speed_x","speed_y"]].head())

,s,dir,speed_x,speed_y
0,0.31,79.43,0.056865,0.304740
1,0.36,118.07,-0.169398,0.317654
2,0.44,130.89,-0.288028,0.332626
3,0.48,134.50,-0.336436,0.342360
4,0.54,129.79,-0.345587,0.414933


## Add Directional Acceleration Features

We also decompose acceleration into directional components so the model can better capture movement changes over time.

In [6]:
df["acc_x"]=df["a"]*np.cos(dir_rad)
df["acc_y"]=df["a"]*np.sin(dir_rad)

display(df[["a","dir","acc_x","acc_y"]].head())

,a,dir,acc_x,acc_y
0,0.49,79.43,0.089884,0.481685
1,0.74,118.07,-0.348207,0.652956
2,0.76,130.89,-0.497503,0.574535
3,0.62,134.50,-0.434564,0.442215
4,0.44,129.79,-0.281589,0.338094


## Add Orientation-Direction Difference

We compute the angular difference between body orientation and movement direction. This feature may help capture turning, reorientation, or reactive movement patterns.

In [7]:
df["dir_minus_o"]=df["dir"]-df["o"]
df["dir_minus_o"] =((df["dir_minus_o"] + 180) % 360)- 180

display(df[["dir","o","dir_minus_o"]].head())

,dir,o,dir_minus_o
0,79.43,267.68,171.75
1,118.07,268.66,-150.59
2,130.89,269.78,-138.89
3,134.50,269.78,-135.28
4,129.79,269.06,-139.27


## Add Momentum Features

We create momentum-style interaction features by combining player weight with directional speed. These features help represents the physical influence of player movement.

In [8]:
df["momentum"]= df["player_weight"]*df["s"]
df["momentum_x"]=df["player_weight"]*df["speed_x"]
df["momentum_y"]=df["player_weight"]*df["speed_y"]

display(df[["player_weight","s","speed_x","speed_y", "momentum","momentum_x","momentum_y"]].head())

,player_weight,s,speed_x,speed_y,momentum,momentum_x,momentum_y
0,204,0.31,0.056865,0.304740,63.24,11.600534,62.166914
1,204,0.36,-0.169398,0.317654,73.44,-34.557187,64.801500
2,204,0.44,-0.288028,0.332626,89.76,-58.757693,67.855664
3,204,0.48,-0.336436,0.342360,97.92,-68.633035,69.841484
4,204,0.54,-0.345587,0.414933,110.16,-70.499712,84.646419


## Add Field-Centered Position Features.

We center player coordinates relative to approximate field midpoints to hel normalize spatial position.

In [9]:
df["x_centered"]=df["x"]-60
df["y_centered"]=df["y"]-26.65

df["dist_from_field_center"]=np.sqrt(df["x_centered"]**2 + df["y_centered"]**2)

display(df[["x","y","x_centered","y_centered","dist_from_field_center"]].head())

,x,y,x_centered,y_centered,dist_from_field_center
0,51.32,20.69,-8.68,-5.96,10.529198
1,51.35,20.66,-8.65,-5.99,10.521530
2,51.39,20.63,-8.61,-6.02,10.505832
3,51.43,20.61,-8.57,-6.04,10.484584
4,51.48,20.58,-8.52,-6.07,10.461133


## Add Simple Temporal Features 

To capture short-term motion dynamics, we create lag and data features within each player trajectory. These features give the model limited temporal context without requiring a full sequence model.


In [10]:
df= df.sort_values(["game_id","play_id","nfl_id","frame_id"]).copy()

group_cols=["game_id","play_id","nfl_id"]

df["x_prev"]=df.groupby(group_cols)["x"].shift(1)
df["y_prev"]=df.groupby(group_cols)["y"].shift(1)
df["s_prev"]=df.groupby(group_cols)["s"].shift(1)
df["a_prev"]=df.groupby(group_cols)["a"].shift(1)

df["delta_x"]=df["x"]-df["x_prev"]
df["delta_y"]=df["x"]-df["y_prev"]
df["delta_s"]=df["x"]-df["s_prev"]
df["delta_a"]=df["x"]-df["a_prev"]

display(
    df[
        ["game_id","play_id","nfl_id","frame_id","x","x_prev","delta_x","s","s_prev","delta_s"]
    ].head()
)

,game_id,play_id,nfl_id,frame_id,x,x_prev,delta_x,s,s_prev,delta_s
52,2023090700,101,44930,1,41.03,NaN,NaN,0.00,NaN,NaN
53,2023090700,101,44930,2,41.03,41.03,0.00,0.00,0.00,41.03
54,2023090700,101,44930,3,41.05,41.03,0.02,0.02,0.00,41.05
55,2023090700,101,44930,4,41.07,41.05,0.02,0.18,0.02,41.05
56,2023090700,101,44930,5,41.11,41.07,0.04,0.57,0.18,40.93


## Handle Missing Values from Temporal Features 

Lag features introduce missing values at the first frame of each player trajectory. For this iteraction, we fill these values with zero to keep the dataset simple and model-ready.

In [11]:
lag_cols=["x_prev","y_prev","s_prev","a_prev","delta_x","delta_y","delta_s","delta_a"]
df[lag_cols]=df[lag_cols].fillna(0)

In [13]:
df["speed_bin"]=pd.qcut(df["s"], q=5, duplicates="drop")
df["speed_bin_code"]=df["speed_bin"].cat.codes
df=df.drop(columns=["speed_bin"])

In [14]:
new_cols_v3 = [
    "dx_to_ball", "dy_to_ball", "dist_to_ball",
    "speed_x", "speed_y",
    "acc_x", "acc_y",
    "dir_minus_o",
    "momentum", "momentum_x", "momentum_y",
    "x_centered", "y_centered", "dist_from_field_center",
    "x_prev", "y_prev", "s_prev", "a_prev",
    "delta_x", "delta_y", "delta_s", "delta_a",
    "speed_bin_code"
]

print("New v3 columns present:")
print([col for col in new_cols_v3 if col in df.columns])
print("Total new v3 columns:", len([col for col in new_cols_v3 if col in df.columns]))

New v3 columns present:
['dx_to_ball', 'dy_to_ball', 'dist_to_ball', 'speed_x', 'speed_y', 'acc_x', 'acc_y', 'dir_minus_o', 'momentum', 'momentum_x', 'momentum_y', 'x_centered', 'y_centered', 'dist_from_field_center', 'x_prev', 'y_prev', 's_prev', 'a_prev', 'delta_x', 'delta_y', 'delta_s', 'delta_a', 'speed_bin_code']
Total new v3 columns: 23


## Save v3 Dataset

We save the enriched v3 dataset so it can be used directly in the next training notebook.

In [15]:
output_path = PROCESSED_DIR / "baseline_train_v3.parquet"
df.to_parquet(output_path, index=False)

print("Saved v3 dataset:", output_path)

Saved v3 dataset: C:\Users\jmontanez\Documents\nfl-big-data-bowl-2026\data\processed\baseline_train_v3.parquet


## Summary

In this notebook, we extended the v2 dataset with:

- target-relative spatial features,
- directional speed and acceleration features,
- orientation-movement difference,
- momentum-based interaction features,
- centered field-position features,
- and simple temporal lag and delta features.

These additions aim to improve the model’s ability to capture motion dynamics, spatial relationships, and short-term trajectory information.

The next step is to train an XGBoost model using this enhanced v3 dataset and compare its performance against earlier iterations.